In [8]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
 
rentals = spark.read.table("silver_rentals")
 
# reporting window: 30 days starting from the first rental day
# (check these two dates look right before going further)
win_start = rentals.agg(F.min("rental_start_date")).first()[0]
win_end = spark.sql(f"select date_add(to_date('{win_start}'), 29)").first()[0]
print("reporting window:", win_start, "to", win_end)
 
ws = F.lit(win_start).cast("timestamp")
we = F.date_add(F.lit(win_end), 1).cast("timestamp")     # end of the last day
 
 
# 1. gold_dim_customer
# tenure is measured up to the end of the reporting window
months = F.months_between(F.lit(win_end), col("registered_on"))
tenure = F.greatest(months / 12, F.lit(0.0))
 
df_dim_customer = spark.read.table("silver_customers") \
    .withColumn("tenure_years", F.round(tenure, 1)) \
    .withColumn("tenure_band",
                F.when(tenure < 1, "0-1 years")
                 .when(tenure < 3, "1-3 years")
                 .when(tenure < 5, "3-5 years")
                 .otherwise("5+ years"))
 
df_dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_customer")
print("gold_dim_customer:", spark.read.table("gold_dim_customer").count())    # 600
 
 
# 2. gold_dim_depot, already clean so promote as is
df_dim_depot = spark.read.table("silver_depots")
 
df_dim_depot.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_depot")
print("gold_dim_depot:", spark.read.table("gold_dim_depot").count())          # 6
 
 
# 3. gold_dim_date, one row per day of the window
df_dim_date = spark.sql(f"select explode(sequence(to_date('{win_start}'), to_date('{win_end}'), interval 1 day)) as date") \
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int")) \
    .withColumn("day_name", F.date_format("date", "EEEE")) \
    .withColumn("week_of_year", F.weekofyear("date")) \
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7).cast("int"))
 
df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_date")
print("gold_dim_date:", spark.read.table("gold_dim_date").count())            # 30
 
 
# 4. gold_fact_rental, every rental kept (returned or not)
# duration is NULL while the machine is still out, zero would claim it came back instantly
duration = (F.unix_timestamp("rental_end_ts") - F.unix_timestamp("rental_start_ts")) / 86400
 
# days inside the window, still-out rentals run to the end of the window
used_start = F.greatest(col("rental_start_ts"), ws)
used_end = F.least(F.coalesce(col("rental_end_ts"), we), we)
in_window = F.greatest((F.unix_timestamp(used_end) - F.unix_timestamp(used_start)) / 86400, F.lit(0.0))
 
df_fact_rental = rentals \
    .withColumn("rental_duration_days", duration) \
    .withColumn("is_returned", col("rental_end_ts").isNotNull().cast("int")) \
    .withColumn("asset_days_in_window", in_window) \
    .withColumn("asset_type", F.regexp_extract("asset_id", r"^[A-Za-z]+", 0)) \
    .select("rental_id", "customer_id", "depot_id", "asset_id", "asset_type", "rental_type",
            "rental_start_ts", "rental_end_ts", "rental_start_date", "rental_end_date",
            "rental_duration_days", "is_returned", "asset_days_in_window")
 
# not partitioned: only 942 rows, partitions would just make lots of tiny files
df_fact_rental.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_fact_rental")
print("gold_fact_rental:", spark.read.table("gold_fact_rental").count())      # 942
 
f = spark.read.table("gold_fact_rental")
print("null durations:", f.filter(col("rental_duration_days").isNull()).count())                      # 175
print("still out but is_returned = 1:", f.filter(col("rental_end_ts").isNull() & (col("is_returned") == 1)).count())   # 0
print("max asset_days_in_window:", f.agg(F.max("asset_days_in_window")).first()[0])                   # <= 30
 
 
# 5. gold_fact_billing
df_fact_billing = spark.read.table("silver_billing") \
    .select("bill_id", "rental_id", "payer_type", "bill_amount", "billing_date")
 
df_fact_billing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_fact_billing")
print("gold_fact_billing:", spark.read.table("gold_fact_billing").count())    # 779
 
 
# 6. optimize + history
spark.sql("OPTIMIZE gold_fact_rental ZORDER BY (depot_id, rental_start_date)")
spark.sql("OPTIMIZE gold_fact_billing ZORDER BY (rental_id)")
 
for t in ["gold_fact_rental", "gold_fact_billing"]:
    print(t)
    spark.sql(f"DESCRIBE HISTORY {t}").select("version", "timestamp", "operation").show(truncate=False)

StatementMeta(, 4f5c9200-0fac-4a2b-b087-1b6f71b30e7b, 10, Finished, Available, Finished, False)

reporting window: 2026-06-01 to 2026-06-30
gold_dim_customer: 600
gold_dim_depot: 6
gold_dim_date: 30
gold_fact_rental: 942
null durations: 175
still out but is_returned = 1: 0
max asset_days_in_window: 18.0
gold_fact_billing: 779
gold_fact_rental
+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|3      |2026-10-22 07:48:13|OPTIMIZE                         |
|2      |2026-10-22 07:48:03|CREATE OR REPLACE TABLE AS SELECT|
|1      |2026-10-22 07:26:18|OPTIMIZE                         |
|0      |2026-10-22 07:26:06|CREATE OR REPLACE TABLE AS SELECT|
+-------+-------------------+---------------------------------+

gold_fact_billing
+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|3      |2026-10-22 07:48:16|

In [9]:
# (a) longest tied-up machine types, returned rentals only
display(spark.sql("""
    SELECT asset_type,
           ROUND(AVG(rental_duration_days), 2) AS avg_duration_days,
           COUNT(*) AS rentals
    FROM gold_fact_rental
    WHERE is_returned = 1
    GROUP BY asset_type
    ORDER BY avg_duration_days DESC
"""))

# (b) fleet utilisation by depot
display(spark.sql("""
    SELECT d.depot_name,
           d.fleet_size,
           ROUND(SUM(r.asset_days_in_window), 1) AS asset_days_used,
           ROUND(SUM(r.asset_days_in_window) / (d.fleet_size * 30) * 100, 1) AS utilisation_pct
    FROM gold_fact_rental r
    JOIN gold_dim_depot d ON r.depot_id = d.depot_id
    GROUP BY d.depot_name, d.fleet_size
    ORDER BY utilisation_pct DESC
"""))
 
# (c) revenue by payer type
display(spark.sql("""
    SELECT payer_type,
           COUNT(*) AS bills,
           ROUND(SUM(bill_amount), 0) AS revenue,
           ROUND(AVG(bill_amount), 0) AS avg_bill
    FROM gold_fact_billing
    GROUP BY payer_type
    ORDER BY revenue DESC
"""))
 
# (d) revenue by depot (billing -> rental -> depot)
display(spark.sql("""
    SELECT d.depot_name,
           COUNT(b.bill_id) AS bills,
           ROUND(SUM(b.bill_amount), 0) AS revenue,
           ROUND(SUM(b.bill_amount) / COUNT(DISTINCT r.rental_id), 0) AS revenue_per_rental
    FROM gold_fact_billing b
    JOIN gold_fact_rental r ON b.rental_id = r.rental_id
    JOIN gold_dim_depot d ON r.depot_id = d.depot_id
    GROUP BY d.depot_name
    ORDER BY revenue DESC
"""))
 
# (e) rental type mix
display(spark.sql("""
    SELECT rental_type,
           COUNT(*) AS rentals,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM gold_fact_rental
    GROUP BY rental_type
    ORDER BY rentals DESC
"""))
 
# (f) machines out right now, by depot (total should be 175)
display(spark.sql("""
    SELECT d.depot_name,
           COUNT(*) AS currently_out
    FROM gold_fact_rental r
    JOIN gold_dim_depot d ON r.depot_id = d.depot_id
    WHERE r.is_returned = 0
    GROUP BY d.depot_name
    ORDER BY currently_out DESC
""")) 

StatementMeta(, 4f5c9200-0fac-4a2b-b087-1b6f71b30e7b, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1dbcd767-8341-4e21-8cb0-7f270e3b500c)

SynapseWidget(Synapse.DataFrame, e1bc013a-c043-4451-9389-48e8f4e58ac6)

SynapseWidget(Synapse.DataFrame, 631c9dc5-082c-495d-9a01-cb4ea82a0bd0)

SynapseWidget(Synapse.DataFrame, 3862700a-4a1c-4518-862e-6204b32f03e0)

SynapseWidget(Synapse.DataFrame, e7a4473c-e69c-417c-8a26-ad87f9947d68)

SynapseWidget(Synapse.DataFrame, 7ae235b4-ed03-4c35-b32b-914fe8c88240)